In [1]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt

from sklearn.model_selection import TimeSeriesSplit

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.preprocessing import (
    OneHotEncoder,
    StandardScaler
)

from sklearn.impute import SimpleImputer

from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    ConfusionMatrixDisplay
)

import joblib

In [2]:
df = pd.read_csv(
    "../data/processed/classification_dataset.csv"
)

df.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_estimated_delivery_date,customer_zip_code_prefix,customer_city,customer_state,number_of_items,...,seller_zip_code_prefix,total_payment_value,number_of_payment_records,average_payment_value,max_installments,order_hour,order_day_of_week,order_month,is_weekend,late_delivery
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-18,3149,sao paulo,SP,1,...,9350,38.71,3.0,12.903333,1.0,10,0,10,0,0
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-08-13,47813,barreiras,BA,1,...,31570,141.46,1.0,141.460000,1.0,20,1,7,0,0
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-09-04,75265,vianopolis,GO,1,...,14840,179.12,1.0,179.120000,3.0,8,2,8,0,0
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-12-15,59296,sao goncalo do amarante,RN,1,...,31842,72.20,1.0,72.200000,1.0,19,5,11,1,0
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-26,9195,santo andre,SP,1,...,8752,28.62,1.0,28.620000,1.0,21,1,2,0,0


In [3]:
df.shape

(96455, 36)

In [4]:
df.columns.tolist()

['order_id',
 'customer_id',
 'order_status',
 'order_purchase_timestamp',
 'order_approved_at',
 'order_estimated_delivery_date',
 'customer_zip_code_prefix',
 'customer_city',
 'customer_state',
 'number_of_items',
 'total_price',
 'total_freight',
 'average_item_price',
 'number_of_unique_products',
 'number_of_unique_sellers',
 'average_product_weight_g',
 'max_product_weight_g',
 'average_product_length_cm',
 'average_product_height_cm',
 'average_product_width_cm',
 'number_of_product_categories',
 'number_of_sellers',
 'seller_state_count',
 'seller_id',
 'seller_city',
 'seller_state',
 'seller_zip_code_prefix',
 'total_payment_value',
 'number_of_payment_records',
 'average_payment_value',
 'max_installments',
 'order_hour',
 'order_day_of_week',
 'order_month',
 'is_weekend',
 'late_delivery']

In [5]:
df["order_purchase_timestamp"] = pd.to_datetime(
    df["order_purchase_timestamp"]
)

In [6]:
df["order_purchase_timestamp"].min()

Timestamp('2016-09-15 12:16:38')

In [7]:
df["order_purchase_timestamp"].max()

Timestamp('2018-08-29 15:00:37')

In [8]:
df["order_purchase_timestamp"].dt.to_period(
    "M"
).value_counts().sort_index()

order_purchase_timestamp
2016-09       1
2016-10     265
2016-12       1
2017-01     748
2017-02    1641
2017-03    2546
2017-04    2303
2017-05    3545
2017-06    3135
2017-07    3872
2017-08    4193
2017-09    4149
2017-10    4478
2017-11    7288
2017-12    5513
2018-01    7069
2018-02    6555
2018-03    7003
2018-04    6798
2018-05    6749
2018-06    6096
2018-07    6156
2018-08    6351
Freq: M, Name: count, dtype: int64

In [9]:
monthly_orders = (
    df.set_index("order_purchase_timestamp")
      .resample("M")
      .size()
)

plt.figure(figsize=(12, 5))

monthly_orders.plot()

plt.title("Orders Over Time")
plt.xlabel("Date")
plt.ylabel("Number of Orders")

plt.show()

ValueError: Invalid frequency: M. Failed to parse with error message: ValueError("'M' is no longer supported for offsets. Please use 'ME' instead.")

In [ ]:
df = df.sort_values(
    "order_purchase_timestamp"
).reset_index(drop=True)

In [ ]:
df[
    [
        "order_purchase_timestamp",
        "late_delivery"
    ]
].head()

In [ ]:
df[
    [
        "order_purchase_timestamp",
        "late_delivery"
    ]
].tail()

In [ ]:
n = len(df)

train_end = int(
    n * 0.70
)

validation_end = int(
    n * 0.85
)

In [ ]:
train_df = df.iloc[
    :train_end
].copy()

validation_df = df.iloc[
    train_end:validation_end
].copy()

test_df = df.iloc[
    validation_end:
].copy()

In [ ]:
print("Train:", train_df.shape)
print("Validation:", validation_df.shape)
print("Test:", test_df.shape)

In [ ]:
print(
    "Train:",
    train_df["order_purchase_timestamp"].min(),
    "→",
    train_df["order_purchase_timestamp"].max()
)

print(
    "Validation:",
    validation_df["order_purchase_timestamp"].min(),
    "→",
    validation_df["order_purchase_timestamp"].max()
)

print(
    "Test:",
    test_df["order_purchase_timestamp"].min(),
    "→",
    test_df["order_purchase_timestamp"].max()
)

In [ ]:
target = "late_delivery"

In [ ]:
X_train = train_df.drop(
    columns=[target]
)

y_train = train_df[target]

In [ ]:
X_validation = validation_df.drop(
    columns=[target]
)

y_validation = validation_df[target]

In [ ]:
X_test = test_df.drop(
    columns=[target]
)

y_test = test_df[target]

In [ ]:
drop_columns = [
    "order_id",
    "customer_id",
    "order_delivered_customer_date",
    "order_delivered_carrier_date",
    "delivery_delay_days",
    "delivery_days",
    "order_estimated_delivery_date"
]

In [ ]:
X_train = X_train.drop(
    columns=drop_columns,
    errors="ignore"
)

X_validation = X_validation.drop(
    columns=drop_columns,
    errors="ignore"
)

X_test = X_test.drop(
    columns=drop_columns,
    errors="ignore"
)

In [ ]:
X_train = X_train.drop(
    columns=["order_purchase_timestamp"],
    errors="ignore"
)

X_validation = X_validation.drop(
    columns=["order_purchase_timestamp"],
    errors="ignore"
)

X_test = X_test.drop(
    columns=["order_purchase_timestamp"],
    errors="ignore"
)

In [ ]:
numeric_features = X_train.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_features = X_train.select_dtypes(
    include=["object"]
).columns.tolist()

In [ ]:
numeric_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="median"
            )
        ),
        (
            "scaler",
            StandardScaler()
        )
    ]
)

In [ ]:
categorical_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="most_frequent"
            )
        ),
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore"
            )
        )
    ]
)

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            numeric_pipeline,
            numeric_features
        ),
        (
            "cat",
            categorical_pipeline,
            categorical_features
        )
    ]
)

In [ ]:
model = Pipeline(
    steps=[
        (
            "preprocessor",
            preprocessor
        ),
        (
            "model",
            RandomForestClassifier(
                n_estimators=300,
                max_depth=15,
                min_samples_split=5,
                min_samples_leaf=2,
                class_weight="balanced",
                random_state=42,
                n_jobs=-1
            )
        )
    ]
)

In [ ]:
model.fit(
    X_train,
    y_train
)

In [ ]:
validation_pred = model.predict(
    X_validation
)

In [ ]:
validation_prob = model.predict_proba(
    X_validation
)[:, 1]

In [ ]:
print(
    "Validation Precision:",
    precision_score(
        y_validation,
        validation_pred,
        zero_division=0
    )
)

print(
    "Validation Recall:",
    recall_score(
        y_validation,
        validation_pred,
        zero_division=0
    )
)

print(
    "Validation F1:",
    f1_score(
        y_validation,
        validation_pred,
        zero_division=0
    )
)

print(
    "Validation ROC-AUC:",
    roc_auc_score(
        y_validation,
        validation_prob
    )
)

In [ ]:
test_pred = model.predict(
    X_test
)

test_prob = model.predict_proba(
    X_test
)[:, 1]

In [ ]:
print(
    "Test Precision:",
    precision_score(
        y_test,
        test_pred,
        zero_division=0
    )
)

print(
    "Test Recall:",
    recall_score(
        y_test,
        test_pred,
        zero_division=0
    )
)

print(
    "Test F1:",
    f1_score(
        y_test,
        test_pred,
        zero_division=0
    )
)

print(
    "Test ROC-AUC:",
    roc_auc_score(
        y_test,
        test_prob
    )
)

In [ ]:
test_results = test_df[
    [
        "order_purchase_timestamp"
    ]
].copy()

test_results["actual"] = y_test.values

test_results["predicted"] = test_pred

test_results["month"] = (
    test_results[
        "order_purchase_timestamp"
    ].dt.to_period("M")
)

In [ ]:
monthly_f1 = (
    test_results
    .groupby("month")
    .apply(
        lambda x: f1_score(
            x["actual"],
            x["predicted"],
            zero_division=0
        )
    )
)

In [ ]:
plt.figure(figsize=(12, 5))

monthly_f1.plot()

plt.title(
    "Monthly F1 Score on Test Data"
)

plt.xlabel("Month")
plt.ylabel("F1 Score")

plt.show()

In [ ]:
rf_model = model.named_steps[
    "model"
]

In [ ]:
trained_preprocessor = (
    model.named_steps[
        "preprocessor"
    ]
)

In [ ]:
feature_names = (
    trained_preprocessor
    .get_feature_names_out()
)

In [ ]:
importances = (
    rf_model.feature_importances_
)

In [ ]:
importance_df = pd.DataFrame({
    "feature": feature_names,
    "importance": importances
})

In [ ]:
importance_df = (
    importance_df
    .sort_values(
        "importance",
        ascending=False
    )
)

In [ ]:
importance_df.head(20)

In [ ]:
top_features = (
    importance_df
    .head(20)
    .sort_values(
        "importance"
    )
)

plt.figure(figsize=(10, 8))

plt.barh(
    top_features["feature"],
    top_features["importance"]
)

plt.xlabel("Importance")
plt.ylabel("Feature")

plt.title(
    "Top Features Influencing Late Delivery Prediction"
)

plt.show()

In [ ]:
sample_index = 10

In [ ]:
sample_order = X_test.iloc[
    [sample_index]
]

In [ ]:
actual_value = y_test.iloc[
    sample_index
]

print(
    "Actual:",
    actual_value
)

In [ ]:
prediction = model.predict(
    sample_order
)[0]

probability = model.predict_proba(
    sample_order
)[0, 1]

print(
    "Prediction:",
    prediction
)

print(
    "Probability of Late:",
    probability
)

In [ ]:
if prediction == 1:
    print(
        "⚠️ Model predicts this order may be late."
    )
else:
    print(
        "✅ Model predicts this order may be on time."
    )

In [ ]:
print(
    f"Late probability: {probability:.2%}"
)